# 13. Jupyter and Git Basics Practice

这一练习覆盖 notebook 结构分析和常见临时文件识别。

In [ ]:
from pathlib import PurePosixPath


In [ ]:
def count_code_cells(notebook):
    return sum(1 for cell in notebook.get('cells', []) if cell.get('cell_type') == 'code')


def list_markdown_headings(notebook):
    headings = []
    for cell in notebook.get('cells', []):
        if cell.get('cell_type') != 'markdown':
            continue
        text = ''.join(cell.get('source', []))
        for line in text.splitlines():
            if line.startswith('#'):
                headings.append(line.lstrip('#').strip())
                break
    return headings


def is_ephemeral_artifact(path):
    pure = PurePosixPath(path)
    parts = set(pure.parts)
    name = pure.name
    return bool(
        parts & {'.ipynb_checkpoints', '__pycache__', '.pytest_cache', 'node_modules', 'tmp'}
        or name.endswith('_backup.ipynb')
        or name.endswith('_backup.md')
        or name == 'README_backup.md'
        or name.startswith('.')
    )


def summarize_notebook(notebook):
    cell_types = {}
    for cell in notebook.get('cells', []):
        key = cell.get('cell_type', 'unknown')
        cell_types[key] = cell_types.get(key, 0) + 1
    return {
        'cells': len(notebook.get('cells', [])),
        'code_cells': count_code_cells(notebook),
        'headings': list_markdown_headings(notebook),
        'cell_types': cell_types,
    }


In [ ]:
def test_count_code_cells():
    notebook = {
        'cells': [
            {'cell_type': 'markdown', 'source': ['# Title\n', 'intro']},
            {'cell_type': 'code', 'source': ['print(1)\n']},
            {'cell_type': 'code', 'source': ['x = 2\n']},
        ]
    }
    assert count_code_cells(notebook) == 2
    print('✅ count_code_cells 通过')


def test_list_markdown_headings():
    notebook = {
        'cells': [
            {'cell_type': 'markdown', 'source': ['# Title\n', 'intro']},
            {'cell_type': 'markdown', 'source': ['## Subheading\n', 'text']},
            {'cell_type': 'code', 'source': ['x = 2\n']},
        ]
    }
    assert list_markdown_headings(notebook) == ['Title', 'Subheading']
    print('✅ list_markdown_headings 通过')


def test_is_ephemeral_artifact():
    assert is_ephemeral_artifact('docs/.ipynb_checkpoints/tmp.ipynb') is True
    assert is_ephemeral_artifact('README_backup.md') is True
    assert is_ephemeral_artifact('00_Prerequisites/intro.md') is False
    print('✅ is_ephemeral_artifact 通过')


def test_summarize_notebook():
    notebook = {
        'cells': [
            {'cell_type': 'markdown', 'source': ['# Hello\n']},
            {'cell_type': 'code', 'source': ['x = 1\n']},
            {'cell_type': 'code', 'source': ['y = 2\n']},
        ]
    }
    summary = summarize_notebook(notebook)
    assert summary['cells'] == 3
    assert summary['code_cells'] == 2
    assert summary['headings'] == ['Hello']
    assert summary['cell_types'] == {'markdown': 1, 'code': 2}
    print('✅ summarize_notebook 通过')


test_count_code_cells()
test_list_markdown_headings()
test_is_ephemeral_artifact()
test_summarize_notebook()


In [ ]:
notebook = {
    'cells': [
        {'cell_type': 'markdown', 'source': ['# Demo\n', 'notes']},
        {'cell_type': 'code', 'source': ['print(1)\n']},
    ]
}
print('摘要:', summarize_notebook(notebook))
print('ignored?:', is_ephemeral_artifact('docs/.pytest_cache/index.html'))
